# 01 · R1: evaluate an official SmolVLA LIBERO checkpoint on LIBERO suites

Runs `POLICY` on every suite in `SUITES`, **10 episodes per task**, fixed seed and initial states.
Tasks are split over the **two T4s** (even/odd ids) and run one task per `lerobot-eval` call,
so finished tasks survive if the session stops; re-running skips them.

- `HuggingFaceVLA/smolvla_libero` (R1): cameras `image`/`image2`, checkpoint default `n_action_steps` = 1.
- `lerobot/smolvla_libero` (R1b, paper architecture): cameras `camera1`/`camera2` → `RENAME = True`; `n_action_steps` = 10.

**How to run:** `kaggle kernels push -p kaggle/01_r1_eval --accelerator NvidiaTeslaT4 -t <seconds>`.
**Outputs** (`/kaggle/working/<TAG>/<suite>/`): `task_<id>/eval_info.json`, `task_<id>/videos/`, `summary.md`, `summary.csv`, `progress.log`.

In [ ]:
# ---- parameters ----
SUITES = ["libero_spatial", "libero_object", "libero_10"]   # libero_goal done (R1 76 %, R1b 88 %)
POLICY, TAG = "lerobot/smolvla_libero", "r1b"
RENAME = True                  # lerobot/smolvla_libero expects camera1/camera2
NAS = 10                       # n_action_steps; 0 = checkpoint default
N_EPISODES = 10
BATCH = 3                      # parallel envs per GPU; 5 x 2 GPUs on 4 CPUs / 31 GB hung or was OOM-killed
SEED = 1000
TASKS = list(range(10))
TASK_TIMEOUT, RETRIES = 2700, 1  # seconds per lerobot-eval call; a hung env pool is killed and retried
open("/tmp/params.sh", "w").write(
    f'SUITES="{" ".join(SUITES)}"\nPOLICY={POLICY}\nTAG={TAG}\nRENAME={int(RENAME)}\nNAS={NAS}\n'
    f'N_EPISODES={N_EPISODES}\nBATCH={BATCH}\nSEED={SEED}\nTASK_TIMEOUT={TASK_TIMEOUT}\nRETRIES={RETRIES}\n'
    f'TASKS_GPU0="{" ".join(map(str, TASKS[0::2]))}"\nTASKS_GPU1="{" ".join(map(str, TASKS[1::2]))}"\n')
print(open("/tmp/params.sh").read())

In [ ]:
%%bash
echo "=== ENV ===" >> /kaggle/working/r1_report.txt
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv | tee -a /kaggle/working/r1_report.txt
echo "CPUs: $(nproc)" | tee -a /kaggle/working/r1_report.txt
free -g | head -2 | tee -a /kaggle/working/r1_report.txt

In [ ]:
%%bash
# System libraries for headless MuJoCo rendering and video decoding
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libegl1 libgl1 libglvnd0 libosmesa6 libosmesa6-dev ffmpeg cmake build-essential > /dev/null 2>&1 \
  && echo "apt: ok" | tee -a /kaggle/working/r1_report.txt || { echo "apt: FAILED" | tee -a /kaggle/working/r1_report.txt; exit 1; }
# LeRobot needs Python >= 3.12; build an isolated venv with uv
pip install -q -U uv
uv python install 3.12 > /dev/null 2>&1
rm -rf /tmp/venv && uv venv /tmp/venv --python 3.12 > /dev/null 2>&1
/tmp/venv/bin/python --version > /dev/null 2>&1 \
  && echo "venv: ok ($(/tmp/venv/bin/python --version))" | tee -a /kaggle/working/r1_report.txt || { echo "venv: FAILED" | tee -a /kaggle/working/r1_report.txt; exit 1; }

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
LOG=/kaggle/working/install.log
cd /tmp && rm -rf lerobot && git clone -q https://github.com/huggingface/lerobot.git && cd lerobot && git checkout -q ba83e4cf43737e2665be0aafcb5f513b83c7d0f3
echo "lerobot commit: $(git rev-parse HEAD)" | tee -a /kaggle/working/r1_report.txt
unset UV_PYTHON UV_PROJECT_ENVIRONMENT PIP_TARGET
# Target the venv explicitly: Kaggle's environment otherwise sends uv to the system Python
uv pip install --python /tmp/venv/bin/python -e ".[smolvla,libero]" > $LOG 2>&1
RC=$?
echo "install exit: $RC ($(head -1 $LOG))" | tee -a /kaggle/working/r1_report.txt
if [ $RC -ne 0 ]; then tail -40 $LOG | tee -a /kaggle/working/r1_report.txt; exit 1; fi
python - <<'EOF' 2>&1 | tee -a /kaggle/working/r1_report.txt
import importlib, torch
for m in ["torch", "lerobot", "transformers", "mujoco", "libero"]:
    mod = importlib.import_module(m)
    print(f"import {m}: ok {getattr(mod, '__version__', '')}")
print("cuda:", torch.version.cuda, "|", torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
EOF
[ ${PIPESTATUS[0]} -eq 0 ] || exit 1
# LIBERO asks for a dataset path on first import; answer "N" to keep defaults
(yes N | python -c "from libero.libero import benchmark; print('libero:', len(benchmark.get_benchmark_dict()), 'suites')") 2>&1 | tail -1 | tee -a /kaggle/working/r1_report.txt
# Kaggle ships its own /usr/local/bin/lerobot-* that shadow the venv's; every call below uses `/tmp/venv/bin/python -m ...`
echo "lerobot-eval on PATH: $(which -a lerobot-eval | tr '\n' ' ')" | tee -a /kaggle/working/r1_report.txt
/tmp/venv/bin/python -c "import lerobot, libero, mujoco" && touch /tmp/setup_ok && echo "setup: ok" | tee -a /kaggle/working/r1_report.txt

In [ ]:
%%bash
# One lerobot-eval call per task so that partial results survive a stopped session
cat > /tmp/run_tasks.sh <<'EOF'
#!/usr/bin/env bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
GPU=$1; shift
ARGS=()
[ "$NAS" != "0" ] && ARGS+=(--policy.n_action_steps=$NAS)
[ "$RENAME" = "1" ] && ARGS+=('--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}')
for SUITE in $SUITES; do
  OUT=/kaggle/working/$TAG/$SUITE
  mkdir -p $OUT
  for T in "$@"; do
    DIR=$OUT/task_$T
    if [ -f $DIR/eval_info.json ]; then echo "[gpu$GPU] task $T: already done" >> $OUT/progress.log; continue; fi
    for TRY in $(seq 0 $RETRIES); do
      rm -rf $DIR
      S=$(date +%s)
      # Kaggle's EGL exposes a single device (id 0) even with both T4s visible, so all rendering
      # goes to EGL device 0; only the policy is placed on cuda:$GPU.
      # timeout: a deadlocked env pool once blocked a GPU for an hour; kill it and retry.
      MUJOCO_EGL_DEVICE_ID=0 timeout -k 30 $TASK_TIMEOUT /tmp/venv/bin/python -m lerobot.scripts.lerobot_eval \
        --policy.path=$POLICY --policy.device=cuda:$GPU "${ARGS[@]}" \
        --env.type=libero --env.task=$SUITE "--env.task_ids=[$T]" --env.max_parallel_tasks=1 \
        --eval.n_episodes=$N_EPISODES --eval.batch_size=$BATCH --seed=$SEED \
        --output_dir=$DIR > $DIR.log 2>&1
      RC=$?
      echo "[gpu$GPU] task $T try $TRY: exit=$RC seconds=$(( $(date +%s) - S )) $(grep -o '"pc_success": [0-9.]*' $DIR/eval_info.json 2>/dev/null | tail -1)" >> $OUT/progress.log
      # results are written before env.close(); a crash on close still leaves a valid eval_info.json
      [ -f $DIR/eval_info.json ] && break
      tail -5 $DIR.log >> $OUT/progress.log
      pkill -f "cuda:$GPU.*task_ids=\[$T\]" 2>/dev/null; sleep 10
    done
  done
done
EOF
chmod +x /tmp/run_tasks.sh
echo runner ready

In [ ]:
%%bash
source /tmp/params.sh
if [ ! -f /tmp/setup_ok ]; then echo "Setup did not finish; see r1_report.txt and install.log"; cat /kaggle/working/r1_report.txt; exit 1; fi
START=$(date +%s)
mkdir -p /kaggle/working/$TAG
# memory/CPU trace every 60 s to diagnose OOM kills (exit 137)
(while true; do echo "$(date +%H:%M:%S) $(free -m | awk '/Mem:/{print "used",$3,"avail",$7}') load $(cut -d' ' -f1 /proc/loadavg)"; sleep 60; done) > /kaggle/working/$TAG/mem.log 2>&1 &
MEMMON=$!
/tmp/run_tasks.sh 0 $TASKS_GPU0 &
P0=$!
sleep 90   # stagger: both runs starting LIBERO at once hung the first pair of tasks
/tmp/run_tasks.sh 1 $TASKS_GPU1 &
P1=$!
wait $P0 $P1
kill $MEMMON 2>/dev/null
echo "=== $TAG done in $(( ($(date +%s) - START) / 60 )) min" | tee -a /kaggle/working/r1_report.txt
for SUITE in $SUITES; do echo "--- $SUITE"; cat /kaggle/working/$TAG/$SUITE/progress.log; done | tee -a /kaggle/working/r1_report.txt

In [ ]:
import json, math, csv
from pathlib import Path

params = {k: v.strip('"') for k, v in (l.strip().split("=", 1) for l in open("/tmp/params.sh") if "=" in l)}

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * (c - h), 100 * (c + h))

for suite in params["SUITES"].split():
    out = Path(f"/kaggle/working/{params['TAG']}/{suite}")
    rows = []
    for f in sorted(out.glob("task_*/eval_info.json"), key=lambda p: int(p.parent.name.split("_")[1])):
        o = json.load(open(f))["overall"]
        k, n = int(o["n_success"]), int(o["n_episodes"])
        lo, hi = wilson(k, n)
        rows.append(dict(task=int(f.parent.name.split("_")[1]), success=k, episodes=n, pc_success=100 * k / n,
                         ci_low=lo, ci_high=hi, sec_per_episode=o.get("eval_ep_s", float("nan"))))
    K, N = sum(r["success"] for r in rows), sum(r["episodes"] for r in rows)
    lo, hi = wilson(K, N)
    lines = [f"# {params['TAG']} · {suite} · {params['POLICY']} · n_action_steps {params['NAS']} · seed {params['SEED']} · {params['N_EPISODES']} ep/task", "",
             "| task | success | SR % | 95% Wilson CI | s/episode |", "|---|---|---|---|---|"]
    lines += [f"| {r['task']} | {r['success']}/{r['episodes']} | {r['pc_success']:.0f} | {r['ci_low']:.0f}–{r['ci_high']:.0f} | {r['sec_per_episode']:.0f} |" for r in rows]
    lines += [f"| **suite** | **{K}/{N}** | **{100 * K / max(N, 1):.1f}** | **{lo:.1f}–{hi:.1f}** | |"]
    if rows:
        (out / "summary.md").write_text("\n".join(lines))
        with open(out / "summary.csv", "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
    print("\n".join(lines))
    print(f"Tasks finished: {len(rows)}/10\n")